In [4]:
# =====================================================================
# SEL 1: INSIALISASI ENVIRONMENT & STANZA (GPU ACCELERATED)
# =====================================================================
import os
import re
import pandas as pd
import stanza
import torch

# Deteksi otomatis ketersediaan perangkat keras
device_hardware = "cuda" if torch.cuda.is_available() else "cpu"
use_gpu_flag = torch.cuda.is_available()

print(f"🖥️ Jalur Komputasi Terdeteksi: [{device_hardware.upper()}]")
print("⏳ Menginisialisasi pipeline NLP Stanza berbasis GPU...")

try:
    # Unduh korpus model bahasa Indonesia jika belum ada
    # stanza.download('id') 
    
    # Inisialisasi Pipeline dengan memaksa penggunaan GPU jika tersedia
    nlp = stanza.Pipeline(
        lang='id', 
        processors='tokenize,pos,lemma', 
        use_gpu=use_gpu_flag,
        logging_level='WARN'
    )
    print(f"✅ Stanza Berhasil Dimuat di -> {device_hardware.upper()}")
except Exception as e:
    print(f"❌ Gagal memuat Stanza dengan Akselerasi GPU. Error: {e}")
    print("🔄 Mencoba fallback menggunakan mode CPU...")
    nlp = stanza.Pipeline(lang='id', processors='tokenize,pos,lemma', use_gpu=False)


# =====================================================================
# SEL 2: LOCK TARGET ASSET BAKU DARI KOLOM 'TIPE' FILE BARU
# =====================================================================
PATH_MASTER_ASET = "../../../data/master_aset_enriched.xlsx"

if os.path.exists(PATH_MASTER_ASET):
    df_master = pd.read_excel(PATH_MASTER_ASET)
    if 'Tipe' in df_master.columns:
        # Ambil unik, buang nan, ubah ke lowercase, dan bersihkan spasi
        tipe_aset_baku = df_master['Tipe'].dropna().unique().tolist()
        tipe_aset_baku = [str(aset).lower().strip() for aset in tipe_aset_baku if str(aset).strip() != ""]
        print(f"🎯 Kunci {len(tipe_aset_baku)} Tipe Aset Baku dari Kolom 'Tipe':")
        print(tipe_aset_baku)
    else:
        raise KeyError(f"Kolom 'Tipe' tidak ditemukan di file {PATH_MASTER_ASET}")
else:
    raise FileNotFoundError(f"File {PATH_MASTER_ASET} belum diunggah ke direktori!")


# =====================================================================
# SEL 3: AMBIL KORPUS KELUHAN UNTUK EKSTRAKSI IMBUHAN
# =====================================================================
PATH_TIKET = "../../../data/dataset_tiket_master_bersih.csv"

if os.path.exists(PATH_TIKET):
    df_tiket = pd.read_csv(PATH_TIKET, sep='|', on_bad_lines='skip')
    korpus_teks = df_tiket['teks_keluhan_awam'].dropna().astype(str).tolist()
    print(f"\n📝 Memuat {len(korpus_teks)} kalimat keluhan untuk mendeteksi variasi bahasa...")
else:
    # Fallback dummy jika dijalankan terpisah tanpa folder data tiket
    korpus_teks = [
        "lantai 1 pompanya bocor parah gedung a",
        "acnya mati tolong diperbaiki di ruang 2",
        "printannya macet kertasnya nyangkut di dalam",
        "lampunya kedap kedip bikin pusing",
        "tolong pcnya mati total ga bisa nyala",
        "kursinya patah di bagian kaki belakang"
    ]
    print("\n⚠️ File data tiket tidak ditemukan, menggunakan data simulasi keluhan.")


# =====================================================================
# SEL 4: AUTOMATED SLANG GENERATOR (PROACTIVE GENERATION)
# =====================================================================
import pandas as pd
import os
import re

kamus_generated = {}

# 1. Daftar imbuhan kasual & formal yang ingin di-generate otomatis
DAFTAR_IMBUHAN = ['nya', 'an', 'nyaa', 'x', 'ny']

# 2. Ambil semua kata tunggal penyusun dari 260 tipe aset baku
kata_dasar_aset = set()
for aset in tipe_aset_baku:
    # Pecah frasa (misal: 'ac split' -> 'ac', 'split')
    for kata in re.findall(r'\b\w+\b', aset):
        # Filter kata sampah/konjungsi/tanda kurung agar kamus tidak rusak
        if kata in ['dan', 'atau', 'with', 'etc', 'non', 'stored', 'pressure', 'system']:
            continue
        # Izinkan kata benda teknis pendek atau kata dengan panjang > 2 karakter
        if len(kata) > 2 or kata in ['ac', 'tv', 'pju', 'grc', 'acp', 'stp', 'lan', 'lsa']:
            kata_dasar_aset.add(kata)

print(f"📦 Berhasil mengisolasi {len(kata_dasar_aset)} kata dasar komponen aset.")
print("⏳ Men-generate seluruh kombinasi imbuhan secara otomatis...")

# 3. Lakukan kombinasi perkalian (Cartesian Product) antara kata dasar dan imbuhan
for kata in kata_dasar_aset:
    for imbuhan in DAFTAR_IMBUHAN:
        slang_word = f"{kata}{imbuhan}"
        kamus_generated[slang_word] = kata

# Pindahkan ke DataFrame dan bersihkan urutan
df_kamus_ekspor = pd.DataFrame(list(kamus_generated.items()), columns=['slang', 'formal'])
df_kamus_ekspor = df_kamus_ekspor.sort_values(by='slang').reset_index(drop=True)

print("\n📊 HASIL GENERASI KAMUS SLANG LENGKAP (PROAKTIF):")
print("="*60)
print(f"Total entri kamus yang berhasil dibuat: {len(df_kamus_ekspor)} baris.")
print("-"*60)
# Tampilkan sampel data acak untuk verifikasi kualitas
print(df_kamus_ekspor.sample(30, random_state=42).to_string(index=False))
print("="*60)

# Simpan hasil akhir ke CSV
OUTPUT_CSV = "data/kamus_slang_aset.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df_kamus_ekspor.to_csv(OUTPUT_CSV, index=False)
print(f"💾 SUKSES! {len(df_kamus_ekspor)} variasi kata berhasil diamankan ke -> {OUTPUT_CSV}")

🖥️ Jalur Komputasi Terdeteksi: [CUDA]
⏳ Menginisialisasi pipeline NLP Stanza berbasis GPU...


2026-05-27 16:14:44 WARNING: Language id package default expects mwt, which has been added


✅ Stanza Berhasil Dimuat di -> CUDA
🎯 Kunci 260 Tipe Aset Baku dari Kolom 'Tipe':
['ac split', 'kipas angin', 'control panel ac', 'lampu wall sign', 'ac cassette', 'lampu esensial', 'genset diesel', 'pompa boster', 'apab dry powder', 'mdp', 'ac portable', 'ac sentral', 'pesawat telepon', 'ac standing', 'indoor capacitor bank', 'lampu downlight', 'ac split duct', 'automatic transfer switch (ats panel)', 'apar dry powder stored pressure', 'apar co2', 'lampu pylon sign', 'lvmdp', 'diesel fire pump', 'jockey pump', 'heat detector', 'smoke detector', 'dvr cctv', 'atap beton', 'dinding keramik/mozaik', 'ac vrv', 'pemipaan air bersih', 'dinding cladding (alumunium composit,/acp,etc)', 'lantai keramik', 'siamese connection', 'atap aspal', 'hydrant pillar', 'exhaust fan', 'sdp', 'pompa transfer', 'pabx', 'wastafel', 'kran air', 'jet shower', 'lampu tl', 'kamera cctv', 'rooftank', 'control panel penerangan', 'ground water tank', 'access control', 'monitor cctv', 'pompa air tanah', 'bell alarm', 